# Reprodução — autoscaling preditivo com LSTM / Bi-LSTM

Versão para **Google Colab**, com Keras e GPU.

Reproduz o desenho de **Dang-Quang & Yoo (2021)**, *Deep Learning-Based Autoscaling
Using Bidirectional LSTM for Kubernetes* — que, aliás, também foi treinado no Colab.

**O que o notebook faz**
1. gera uma série de carga por minuto (ciclo diário, semanal, rajadas)
2. monta janelas de 10 minutos, com divisão **temporal** 70/30
3. treina LSTM e Bi-LSTM em Keras
4. compara com baselines simples
5. converte a carga prevista em **número de pods** (Algoritmo 1 do artigo)
6. mede elasticidade pelas métricas do SPEC

> Em `src/` do repositório há a mesma coisa em NumPy puro, para rodar sem instalar nada.


## 0. Ambiente


In [ ]:
import sys, platform, numpy as np, tensorflow as tf
print('python  ', platform.python_version())
print('numpy   ', np.__version__)
print('tf      ', tf.__version__)
print('GPU     ', tf.config.list_physical_devices('GPU') or 'nenhuma (CPU serve: o modelo é pequeno)')


## 1. Série de carga

Mesma lógica de `src/gerar_trace.py`. Para usar dados reais, troque esta célula pela
leitura do traço **NASA** ou **FIFA 1998** agregado por minuto.


In [ ]:
import math, random
from datetime import datetime, timedelta, timezone

SEMANAS, SEED = 4, 42
REQ_BASE, REQ_PICO = 1200.0, 9000.0

def perfil_diario(h):
    manha = math.exp(-((h-10.0)**2)/(2*2.2**2))
    tarde = math.exp(-((h-15.5)**2)/(2*2.8**2))
    noite = 0.25*math.exp(-((h-21.0)**2)/(2*1.8**2))
    return min(1.0, manha + 0.9*tarde + noite)

SEMANAL = [1.00, 1.03, 1.02, 1.00, 0.95, 0.55, 0.45]

rnd = random.Random(SEED)
total = SEMANAS*7*24*60
rajadas = {}
for _ in range(SEMANAS*6):
    ini, dur, amp = rnd.randrange(total), rnd.randint(3,12), rnd.uniform(1.6,3.2)
    for k in range(dur):
        if ini+k < total:
            forma = math.sin(math.pi*(k+0.5)/dur)
            rajadas[ini+k] = max(rajadas.get(ini+k,1.0), 1.0+(amp-1.0)*forma)

inicio = datetime(2026,1,5,0,0,tzinfo=timezone.utc)
serie, carimbos = [], []
for i in range(total):
    ts = inicio + timedelta(minutes=i)
    v = REQ_BASE + (REQ_PICO-REQ_BASE)*perfil_diario(ts.hour + ts.minute/60)
    v *= SEMANAL[ts.weekday()]
    v *= 1.0 + 0.0009*(i/(24*60))
    v *= rajadas.get(i, 1.0)
    v *= rnd.gauss(1.0, 0.06)
    serie.append(max(0.0, v)); carimbos.append(ts)
serie = np.array(serie)
print(f'{len(serie)} minutos | min={serie.min():.0f} | média={serie.mean():.0f} | máx={serie.max():.0f} req/min')


In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2, 1, figsize=(13,6))
ax[0].plot(serie, lw=0.4); ax[0].set_title('Série completa (4 semanas)'); ax[0].set_ylabel('req/min')
ax[1].plot(serie[:3*24*60], lw=0.8); ax[1].set_title('Três primeiros dias'); ax[1].set_ylabel('req/min')
plt.tight_layout(); plt.show()


## 2. Janelas e divisão temporal

⚠️ A divisão é **por tempo**, nunca aleatória: embaralhar série temporal vaza o futuro.


In [ ]:
JANELA, HORIZONTE, FRACAO = 10, 1, 0.70

def montar_janelas(s, janela=JANELA, horizonte=HORIZONTE):
    n = len(s) - janela - horizonte + 1
    X = np.stack([s[i:i+janela] for i in range(n)])[..., None]
    y = np.array([s[i+janela+horizonte-1] for i in range(n)])[:, None]
    return X, y

corte = int(len(serie)*FRACAO)
vmin, vmax = serie[:corte].min(), serie[:corte].max()   # normaliza SÓ com o treino
serie_n = (serie - vmin)/(vmax - vmin)

X, y = montar_janelas(serie_n)
corte_j = corte - JANELA - HORIZONTE + 1
X_tr, y_tr, X_te, y_te = X[:corte_j], y[:corte_j], X[corte_j:], y[corte_j:]
y_real = serie[JANELA+HORIZONTE-1:][corte_j:]
print('treino', X_tr.shape, '| teste', X_te.shape)


## 3. Modelos em Keras

Mesma configuração do artigo: 30 unidades ocultas, perda MSE, parada antecipada.


In [ ]:
from tensorflow.keras import Sequential, layers, callbacks

def construir(bidirecional=False, unidades=30):
    nucleo = layers.LSTM(unidades, activation='tanh')
    rec = layers.Bidirectional(nucleo) if bidirecional else nucleo
    m = Sequential([layers.Input((JANELA,1)), rec, layers.Dense(1)])
    m.compile(optimizer='adam', loss='mse')
    return m

modelos, previsoes = {}, {}
for bidir in (False, True):
    nome = 'Bi-LSTM' if bidir else 'LSTM'
    print('\n===', nome)
    m = construir(bidir)
    m.fit(X_tr, y_tr, validation_split=0.1, epochs=50, batch_size=64, verbose=2,
          callbacks=[callbacks.EarlyStopping(patience=5, restore_best_weights=True)])
    p = m.predict(X_te, verbose=0)*(vmax-vmin) + vmin
    modelos[nome], previsoes[nome] = m, p.ravel()


## 4. Qualidade da previsão


In [ ]:
def metricas_previsao(real, prev):
    real, prev = np.ravel(real), np.ravel(prev)
    e = real - prev
    mse = float(np.mean(e**2))
    return {'MSE': mse, 'RMSE': float(np.sqrt(mse)), 'MAE': float(np.mean(abs(e))),
            'MAPE_%': float(np.mean(abs(e/np.where(real==0, np.nan, real)))*100),
            'R2': float(1 - np.sum(e**2)/np.sum((real-real.mean())**2))}

previsoes['Persistência'] = (X_te[:,-1,0])*(vmax-vmin) + vmin
previsoes['Média móvel (5)'] = X_te[:,-5:,0].mean(axis=1)*(vmax-vmin) + vmin

import pandas as pd
tabela = pd.DataFrame({n: metricas_previsao(y_real, p) for n, p in previsoes.items()}).T
tabela.sort_values('RMSE')


## 5. Da carga prevista para os pods

Algoritmo 1 do artigo: se sobe, aplica direto; se desce, remove só uma fração
(`RRS`) do excedente, para reagir mais rápido à próxima rajada.


In [ ]:
CAPACIDADE_POD, PODS_MIN, PODS_MAX, RRS, CDT = 500.0, 10, 60, 0.60, 1

def necessarios(carga):
    return max(PODS_MIN, min(PODS_MAX, math.ceil(carga/CAPACIDADE_POD)))

def simular_preditivo(prev, real):
    forn, req, atual, desde, mud = [], [], PODS_MIN, CDT, 0
    for pv, rl in zip(prev, real):
        req.append(necessarios(rl))
        if desde >= CDT:
            alvo = necessarios(pv)
            if alvo > atual: atual, desde, mud = alvo, 0, mud+1
            elif alvo < atual:
                exced = math.floor((atual-alvo)*RRS)
                if exced > 0: atual, desde, mud = max(PODS_MIN, atual-exced), 0, mud+1
        else: desde += 1
        forn.append(atual)
    return forn, req, mud

def simular_reativo(real, alvo_uso=0.70, atraso=1, janela=5):
    forn, req, atual, hist, mud = [], [], PODS_MIN, [], 0
    for t, rl in enumerate(real):
        req.append(necessarios(rl))
        obs = real[max(0, t-atraso)]
        alvo = max(PODS_MIN, min(PODS_MAX, math.ceil(atual*(obs/(atual*CAPACIDADE_POD))/alvo_uso)))
        hist.append(alvo)
        if alvo > atual: atual, mud = alvo, mud+1
        elif alvo < atual and len(hist) >= janela and max(hist[-janela:]) < atual:
            atual, mud = max(hist[-janela:]), mud+1
        forn.append(atual)
    return forn, req, mud


## 6. Elasticidade (métricas do SPEC)

`θ_U` quanto faltou · `θ_O` quanto sobrou · `T_U`/`T_O` por quanto tempo. Menor é melhor.


In [ ]:
def elasticidade(req, forn):
    r, p = np.array(req, float), np.array(forn, float)
    rs = np.where(r<=0, np.nan, r)
    return {'θ_U_%': float(np.nanmean(np.maximum(r-p,0)/rs)*100),
            'θ_O_%': float(np.nanmean(np.maximum(p-r,0)/rs)*100),
            'T_U_%': float(np.mean(r>p)*100), 'T_O_%': float(np.mean(p>r)*100)}

linhas, series_pods = {}, {}
req0 = [necessarios(c) for c in y_real]
linhas['Sem autoscaler (fixo no máximo)'] = elasticidade(req0, [PODS_MAX]*len(y_real))
series_pods['Sem autoscaler'] = [PODS_MAX]*len(y_real)

f, r, mud = simular_reativo(list(y_real))
linhas['Reativo (tipo HPA)'] = {**elasticidade(r,f), 'mudanças': mud}
series_pods['Reativo (tipo HPA)'] = f

for nome in ('LSTM','Bi-LSTM'):
    f, r, mud = simular_preditivo(list(previsoes[nome]), list(y_real))
    linhas[f'Preditivo ({nome})'] = {**elasticidade(r,f), 'mudanças': mud}
    series_pods[f'Preditivo ({nome})'] = f

pd.DataFrame(linhas).T


## 7. Gráficos


In [ ]:
fatia = slice(0, 2*24*60)
fig, ax = plt.subplots(2,1, figsize=(14,7), sharex=True)
ax[0].plot(y_real[fatia], label='carga real', lw=1, color='black')
for nome in ('LSTM','Bi-LSTM'):
    ax[0].plot(previsoes[nome][fatia], label=f'previsão {nome}', lw=0.9, alpha=.8)
ax[0].set_ylabel('req/min'); ax[0].legend(); ax[0].set_title('Previsão de carga')

ax[1].plot([necessarios(c) for c in y_real[fatia]], label='pods necessários', color='black', lw=1)
for nome, s in series_pods.items():
    if nome != 'Sem autoscaler': ax[1].step(range(len(s[fatia])), s[fatia], where='post', label=nome, alpha=.8)
ax[1].set_ylabel('pods'); ax[1].set_xlabel('minuto'); ax[1].legend(); ax[1].set_title('Decisão de escala')
plt.tight_layout(); plt.show()


## Próximos passos

1. **Trocar a série sintética pelos traços reais** NASA e FIFA 1998 (Internet Traffic Archive), agregados por minuto
2. Acrescentar **Holt–Winters** e o **perfil histórico por horário** aos baselines
3. Medir o **tempo de predição** — é critério de projeto para um controlador de 15 s
4. Ligar no cluster real: substituir a simulação pela chamada à API do Kubernetes
5. **Medir energia com RAPL** — é aqui que a dissertação sai do estado da arte
